# Field Scan : validation croisée par ferme du DeepForest mixte (Kaggle)

À chaque tour, une des 5 fermes (hors Nestor) est mise de côté. Un **modèle neuf** apprend sur
**Count banana plants (vols 0221 + 0222) + les 4 autres fermes ReforesTree**, puis il est noté sur la ferme mise de côté.
Nestor (test final) n'est jamais utilisé.

**Réglage honnête** : pour noter une ferme, le seuil et le nms sont choisis sur les 4 **autres** fermes
(même règle que pour la méthode simple). Le bilan compare ferme par ferme avec la méthode simple.

Durée : environ 1 h 30 à 2 h. Pour pouvoir fermer le navigateur : **Save Version → Save & Run All (Commit)** ;
les résultats seront dans l'onglet Output de la version. Avant : GPU T4, Internet On,
Input `count_banana_v15` et `tuiles_reforestree`. Résultats : `/kaggle/working/validation_croisee_mixte/`.


In [ ]:
# 1. Installation et repérage des données
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # n'utiliser qu'une des deux cartes graphiques
!pip install -q "deepforest>=2,<3"
from pathlib import Path
import torch
ENTREE = Path('/kaggle/input')
coco = sorted(ENTREE.rglob('_annotations.coco.json'))
tuiles_csv = sorted(ENTREE.rglob('tuiles.csv'))
print(len(coco), 'fichiers COCO trouvés (attendu : 3) | tuiles.csv :', tuiles_csv)
assert len(coco) == 3 and len(tuiles_csv) == 1, "Données manquantes : vérifier les deux datasets en Input"
print('GPU :', torch.cuda.is_available())


In [ ]:
# 2. Préparation : les deux jeux dans un même dossier d'images, un seul fichier d'apprentissage
import json, re, time, shutil
import numpy as np, pandas as pd
from PIL import Image

IMAGES = Path('/kaggle/working/images'); IMAGES.mkdir(parents=True, exist_ok=True)
LABEL = 'Tree'

# 2a. Count banana plants : réduction 3 cm -> 4,5 cm (taille d'origine estimée), boîtes au format DeepForest
PIXEL_ORIGINE, PIXEL_CIBLE = 0.03, 0.045
F = PIXEL_ORIGINE / PIXEL_CIBLE
cb_boites, cb_images = [], []
for fichier in coco:
    j = json.load(open(fichier)); par_image = {}
    for a in j['annotations']:
        par_image.setdefault(a['image_id'], []).append(a['bbox'])
    for im in j['images']:
        nom = im['file_name'].split('_JPG')[0] + '.png'
        img = Image.open(fichier.parent / im['file_name']).convert('RGB')
        img.resize((round(img.width * F), round(img.height * F)), Image.Resampling.LANCZOS).save(IMAGES / nom)
        cb_images.append({'image_path': nom, 'vol': re.match(r'(100_\d+)', nom)[1]})
        for x, y, w, h in par_image.get(im['id'], []):
            if w * F >= 2 and h * F >= 2:
                cb_boites.append({'image_path': nom, 'xmin': round(x * F), 'ymin': round(y * F),
                                  'xmax': round((x + w) * F), 'ymax': round((y + h) * F), 'label': LABEL})
cb_boites, cb_images = pd.DataFrame(cb_boites), pd.DataFrame(cb_images)
cb_app = cb_images[cb_images.vol.isin(['100_0221', '100_0222'])].image_path
cb_val = cb_images[cb_images.vol.isin(['100_0223', '100_0224'])].image_path

# 2b. ReforesTree : tuiles copiées dans le même dossier, bananiers seulement, tuiles sans bananier gardées comme exemples vides
TUILES = tuiles_csv[0].parent
tuiles = pd.read_csv(TUILES / 'tuiles.csv')
ann = pd.read_csv(TUILES / 'annotations.csv')
tuiles['y0'] = tuiles.image_path.str.extract(r'_(\d+)_\d+\.png$')[0].astype(int)
tuiles['x0'] = tuiles.image_path.str.extract(r'_(\d+)\.png$')[0].astype(int)
for n in tuiles.image_path:
    shutil.copy(TUILES / 'images' / n, IMAGES / n)
FERMES = ['Flora Pluas RGB', 'Leonor Aspiazu RGB', 'Carlos Vera Guevara RGB',
          'Carlos Vera Arteaga RGB', 'Manuel Macias RGB']          # Nestor Macias = test, exclu
COLONNES = ['image_path', 'xmin', 'ymin', 'xmax', 'ymax', 'label']
cb_app_boites = cb_boites[cb_boites.image_path.isin(cb_app)][COLONNES]

def fichier_apprentissage(fermes, chemin):
    noms = tuiles[tuiles.site.isin(fermes)].image_path
    b = ann[(ann.label == 'banane') & ann.image_path.isin(noms)].copy()
    b = b[(b.xmax > b.xmin) & (b.ymax > b.ymin)]
    b['label'] = LABEL
    vides = pd.DataFrame({'image_path': sorted(set(noms) - set(b.image_path)),
                          'xmin': 0, 'ymin': 0, 'xmax': 0, 'ymax': 0, 'label': LABEL})
    pd.concat([cb_app_boites, b[COLONNES], vides[COLONNES]]).to_csv(chemin, index=False)
    return len(noms)

def sans_chevauchement(site):
    t = tuiles[tuiles.site == site]
    return t[(t.y0 % 400 == 0) & (t.x0 % 400 == 0)]
print(len(cb_app), 'images Count banana en apprentissage à chaque tour')


In [ ]:
# 3. Outils de notation (règle du sujet : centre à moins de 1 m d'un vrai bananier)
from scipy.optimize import linear_sum_assignment

RAYON_M = 1.0
SEUILS_CONFIANCE = np.round(np.arange(0.05, 0.96, 0.05), 2)
SEUILS_NMS = [0.05, 0.2, 0.4]

def centres(df):
    return np.c_[(df.xmin + df.xmax) / 2, (df.ymin + df.ymax) / 2]

def apparier(det, vrai, rayon_px):
    if len(det) == 0 or len(vrai) == 0:
        return 0
    d = np.linalg.norm(det[:, None] - vrai[None], axis=2)
    cout = np.where(d <= rayon_px, d, 1e6)
    l, c = linear_sum_assignment(cout)
    return int((cout[l, c] <= rayon_px).sum())

def grille(modele, noms, verite, taille_pixel_m):
    # noms : images à noter (dans IMAGES) ; verite : boîtes vraies ; taille_pixel_m : un nombre par image
    lignes = []
    for nms in SEUILS_NMS:
        modele.model.nms_thresh = nms
        modele.model.score_thresh = 0.05
        pred = {}
        for nom in noms:
            p = modele.predict_image(path=str(IMAGES / nom))
            pred[nom] = p if p is not None else pd.DataFrame(columns=['xmin', 'ymin', 'xmax', 'ymax', 'score'])
        for s in SEUILS_CONFIANCE:
            vp = nb = 0
            for nom in noms:
                p = pred[nom]; p = p[p.score >= s]
                vp += apparier(centres(p), centres(verite[verite.image_path == nom]), RAYON_M / taille_pixel_m[nom])
                nb += len(p)
            lignes.append({'nms': nms, 'seuil': s, 'detections': nb, 'bananes_vraies': len(verite), 'vp': vp})
    g = pd.DataFrame(lignes)
    g['precision'] = g.vp / g.detections.clip(lower=1)
    g['rappel'] = g.vp / g.bananes_vraies
    g['f1'] = 2 * g.vp / (g.detections + g.bananes_vraies)
    return g


In [ ]:
# 4. Les 5 tours (un modèle neuf à chaque tour)
from deepforest import main
import gc
try:
    from lightning.pytorch.loggers import CSVLogger
except ImportError:
    from pytorch_lightning.loggers import CSVLogger

EPOQUES = 10
SORTIE = Path('/kaggle/working/validation_croisee_mixte'); SORTIE.mkdir(parents=True, exist_ok=True)

for ferme in FERMES:
    nom = ferme.replace(' RGB', '').replace(' ', '_')
    if (SORTIE / f'grille_{nom}.csv').exists():
        print(f'{ferme} : déjà fait'); continue
    autres = [f for f in FERMES if f != ferme]
    n_tuiles = fichier_apprentissage(autres, Path('/kaggle/working') / f'app_{nom}.csv')
    validation = sans_chevauchement(ferme)
    print(f'\n=== Ferme notée : {ferme} ({len(validation)} tuiles) | apprentissage : Count banana + {autres} ({n_tuiles} tuiles)')

    torch.manual_seed(0)
    modele = main.deepforest()
    modele.load_model('weecology/deepforest-tree')
    modele.config.train.csv_file = str(Path('/kaggle/working') / f'app_{nom}.csv')
    modele.config.train.root_dir = str(IMAGES)
    modele.config.train.epochs = EPOQUES
    modele.config.batch_size = 8
    modele.config.workers = 2
    journal = CSVLogger('/kaggle/working/journaux', name=nom)
    debut = time.time()
    modele.create_trainer(logger=journal)
    modele.trainer.fit(modele)
    print(f'Apprentissage : {(time.time() - debut) / 60:.1f} min')

    j = pd.read_csv(Path(journal.log_dir) / 'metrics.csv')
    j.to_csv(SORTIE / f'journal_{nom}.csv', index=False)
    pertes = j.dropna(subset=['train_loss_epoch']) if 'train_loss_epoch' in j else j.iloc[0:0]
    if j.epoch.max() + 1 < EPOQUES or len(pertes) < EPOQUES or not np.isfinite(pertes.train_loss_epoch).all():
        print(f'!!! {ferme} : apprentissage incomplet ou loss NaN, tour NON enregistré')
        del modele; gc.collect(); torch.cuda.empty_cache(); continue
    print('loss par époque :', ' '.join(f'{x:.2f}' for x in pertes.train_loss_epoch))

    verite = ann[(ann.label == 'banane') & ann.image_path.isin(validation.image_path)]
    g = grille(modele, list(validation.image_path), verite, dict(zip(validation.image_path, validation.taille_pixel_m)))
    g.insert(0, 'ferme', ferme)
    g.to_csv(SORTIE / f'grille_{nom}.csv', index=False)
    print(f'{ferme} : meilleur F1 possible {g.f1.max():.3f}')
    del modele; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# 5. Bilan honnête : chaque ferme notée avec le réglage choisi sur les 4 AUTRES fermes
grilles = pd.concat([pd.read_csv(f) for f in sorted(SORTIE.glob('grille_*.csv'))])
methode_simple = {'Flora Pluas RGB': 0.410, 'Leonor Aspiazu RGB': 0.232, 'Carlos Vera Guevara RGB': 0.165,
                  'Carlos Vera Arteaga RGB': 0.119, 'Manuel Macias RGB': 0.204}
lignes = []
for ferme in grilles.ferme.unique():
    autres = grilles[grilles.ferme != ferme].groupby(['nms', 'seuil'])[['detections', 'bananes_vraies', 'vp']].sum()
    nms, seuil = (2 * autres.vp / (autres.detections + autres.bananes_vraies)).idxmax()
    r = grilles[(grilles.ferme == ferme) & (grilles.nms == nms) & (grilles.seuil == seuil)].iloc[0]
    lignes.append({'ferme': ferme, 'nms': nms, 'seuil': seuil, 'bananes_vraies': r.bananes_vraies,
                   'detections': r.detections, 'vp': r.vp, 'precision': r.precision, 'rappel': r.rappel,
                   'f1_mixte': r.f1, 'f1_methode_simple': methode_simple[ferme]})
bilan = pd.DataFrame(lignes)
pd.set_option('display.width', 200)
print(bilan.round(3).to_string(index=False))
t = bilan[['detections', 'bananes_vraies', 'vp']].sum()
print(f"\nF1 global DeepForest mixte : {2 * t.vp / (t.detections + t.bananes_vraies):.3f} "
      f"(précision {t.vp / t.detections:.3f}, rappel {t.vp / t.bananes_vraies:.3f})")
print('F1 global méthode simple (même règle) : 0.247')
bilan.to_csv(SORTIE / 'bilan.csv', index=False)
